In [1]:
import os

os.environ.pop("CURL_CA_BUNDLE", None)

print("CURL_CA_BUNDLE =", os.environ.get("CURL_CA_BUNDLE"))

CURL_CA_BUNDLE = None


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
import wikipedia
from langchain_community.tools import WikipediaQueryRun, DuckDuckGoSearchResults
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.tools import tool

from dotenv import load_dotenv
import os
load_dotenv()


if os.environ['GEMINI_API_KEY']:
    print('gemini key set')
else:
    print("not set")

gemini key set


# **Initialize LLM**

In [22]:
# Initialize the Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16', 'langchain-google-genai': '4.3.5'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', max_retries=2, client=<google.genai.client.Client object at 0x000002257F3E5160>, default_metadata=(), model_kwargs

In [19]:

@tool
def duck_tool(query: str):
    """"This tool allows you to search duck duck go for information on a given topic"""
    tool = DuckDuckGoSearchResults()

    return tool.invoke(query)
    
    
# result = duck_tool.invoke({"query": "what are the Trending news in kenya about Linda Mwananchi"})
# print(result)

result = duck_tool.invoke({"query": "What is LangChain?"})
print(result)

snippet: 2 weeks ago - LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications., title: LangChain - Wikipedia, link: https://en.wikipedia.org/wiki/LangChain, snippet: 14 hours ago - What is LangChain how and why businesses use LangChain, and how to use LangChain with AWS., title: What is LangChain? - LangChain Explained - AWS, link: https://aws.amazon.com/what-is/langchain/, snippet: August 26, 2025 - LangChain is a framework for building AI applications that are connected with other data sources and tools., title: r/LangChain on Reddit: can someone explain Langchain in a simple manner, link: https://www.reddit.com/r/LangChain/comments/1n0qam7/can_someone_explain_langchain_in_a_simple_manner/, snippet: 1 month ago - LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware., title: LangChain overview - Docs by Lan

## **Arxiv Retriever**

In [ ]:
from langchain_community.retrievers import ArxivRetriever
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper

arxiv_query = ArxivQueryRun(api_wrapper=ArxivAPIWrapper())

arxiv_query.invoke("Transformers in NLP")

# retriever = ArxivRetriever( load_max_docs=2, get_ful_documents=True, )

# docs = retriever.invoke("Transformers in NLP")


# **Wikipedia Search Tool**

In [10]:


wikipedia.set_user_agent(
    "LangGraphTutorial/1.0 (educational project)"
)

@tool
def wiki_tool(query: str):
    """"This tool allows you to search wikipedia for information on a given topic"""
    wiki = WikipediaAPIWrapper()

    wiki_query = WikipediaQueryRun(api_wrapper=wiki)

    result = wiki_query.invoke(query)
    # print(result)
    return result

result = wiki_tool.invoke({"query": "What is LangChain?"})
print(result)

Page: List of artificial intelligence companies
Summary: Below is a list of notable companies that primarily focus on artificial intelligence (AI). Companies that simply make use of AI but have a different primary focus are not included.

Page: Agent harness
Summary: An agent harness is the software infrastructure surrounding a large language model (LLM) that enables it to operate as an AI agent: it manages tool use, memory, state persistence, execution environments and feedback loops, as opposed to the model's own reasoning. A shorthand popularised in 2026 expresses the relationship as Agent = Model + Harness.
Because an LLM is stateless and, unaided, produces only text, the harness is what allows a model to take actions over multiple steps, use external tools, and sustain a long-running task across sessions. Rather than repeatedly re-reading an ever-growing transcript inside the context window, a harness can offload record-keeping into a structured software environment that manages t

## **Custom Tool**

In [30]:


@tool
def personal_info(name: str):
    """
    use this tool to get personal information about Alice, Bob and Charlie.
    """
    
    info = {
        "Alice":"Alice is a software engineer with 5 years experience in AI",
        "Bob":"Bob is a data scientist. Loves working with large datasets",
        "Charlie":"Charlie is a product manager with a background in tech startups"
    }
    
    return info.get(name, "no information available for this person")

personal_info.invoke("Alice")

'Alice is a software engineer with 5 years experience in AI'

## **Tool Binding**

In [34]:
tools = [duck_tool, wiki_tool, personal_info]

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What is LangChain?")

response.tool_calls

[{'name': 'duck_tool',
  'args': {'query': 'what is LangChain'},
  'id': 'call_252335',
  'type': 'tool_call'}]